# Load Libs

In [1]:
%pip install wfdb

import ast
import pandas as pd
import wfdb
from sklearn.preprocessing import MultiLabelBinarizer
import torch
import os
import numpy as np

from metriclib.metric import StreamMetric, TabularMetric, MetricResult
from metriclib.report import Report

from metriclib.data import Dataset

Note: you may need to restart the kernel to use updated packages.


# Load Dataset

In [2]:
base_dir = os.path.join("..", "sample-data")
agg_path = os.path.join(base_dir, "scp_statements.csv")
aggregation_df = pd.read_csv(agg_path, index_col=0)
diag_agg_df = aggregation_df[aggregation_df.diagnostic == 1.0]


def aggregate_diagnostic(y_dic, diag_agg_df=None):
    tmp = []
    for key in y_dic.keys():
        if key in diag_agg_df.index:
            c = diag_agg_df.loc[key].diagnostic_class
            if str(c) != "nan":
                tmp.append(c)
    return list(set(tmp))


class PTBXLDataset(Dataset):
    def __init__(self):
        self.metadata = pd.read_csv(os.path.join(base_dir, "ptbxl_subsample.csv"))
        
        self.labels = MultiLabelBinarizer().fit_transform(
            self.metadata["scp_codes"].apply(
                lambda x: aggregate_diagnostic(ast.literal_eval(x), diag_agg_df)
            ),
        )

    def __len__(self):
        return len(self.metadata.index)

    def __getitem__(self, index):
        metadata = self.metadata.iloc[index].to_dict()
        metadata["idx"] = index
        record_path = os.path.join(base_dir, self.metadata["filename_lr"].iloc[index])
        x = wfdb.rdsamp(record_path)[0].T
        y = torch.tensor(self.labels[index])

        return x, y, metadata


dataset1 = PTBXLDataset()

# Adding a Custom Metric

## Tabular Metric

In [3]:
class MeanAge(TabularMetric):
    def compute(self, data, reference, metric_config):
        return {
            "description": "Mean age of patients",
            "value": data["age"].mean(),
        }

## Streaming Metric

In [ ]:
import sys
import antropy as ant


class Entropy(StreamMetric):

    def aggregate(self, datapoint, reference=None, metric_config=None):

        values = []
        for i in range(datapoint[0].shape[0]):
            x = np.asarray(datapoint[0][i, :], dtype=np.float64)
            x = np.ascontiguousarray(x)  # important for numba
            m = int(2)
            r = float(0.2 * np.std(x))
            values.append(
                ant.sample_entropy(x, order=m, tolerance=r, metric="chebyshev")
            )

        entropy = float(np.mean(values))
        return entropy

    def compute(self, data, reference, metric_config):
        return MetricResult(
            cluster=None,
            threshold=0,
            description="Mean sample entropy across all leads",
            value=np.array(data).mean(),
        )

# Create Report

In [5]:
report = Report(datasets=[dataset1])
report.add_metric(
    "HillNumbers",
    metric_config={"column": "device", "types": list(set(dataset1.metadata["device"]))},
)

report.add_metric("Entropy")
report.generate()

100%|██████████| 10/10 [00:00<00:00, 62.44it/s]


([{'metric': metriclib.metrics.representativeness.HillNumbers,
   'reference': None,
   'metric_config': {'column': 'device', 'types': ['CS-12   E']},
   'dataset': 'PTBXLDataset',
   'name': None,
   'result': MetricResult(description='Hill Numbers q=2 for device', value=np.float64(1.0), cluster='Representativeness', threshold=1)},
  {'metric': __main__.Entropy,
   'reference': None,
   'metric_config': None,
   'dataset': 'PTBXLDataset',
   'name': None,
   'result': MetricResult(description='Mean sample entropy across all leads', value=np.float64(0.30223730917444913), cluster=None, threshold=0)}],
 [],
 [{'Measurement Process': 0,
   'Timeliness': 0,
   'Representativeness': np.float64(1.0),
   'Informativeness': 0,
   'Consistency': 0}])